# PixRecon: Pix statement reconciliation benchmark (73 cases)

A merchant's Pix bank statement must be reconciled against its order list: duplicated exports, refunds (full and partial), late and partial payments, overpayments, orphan transfers, amounts written three ways, timestamps in UTC and -03:00. The model answers one strict JSON object per case; a deterministic grader (no LLM judge) scores it.

* 73 cases: 20 easy, 20 medium, 20 hard (synthetic, seeded) + 13 gold edge cases (deadline boundaries, BRL cents, timezone instants, refund arithmetic).
* Score per case = content score (0 to 1) if the answer follows the exact schema, else 0. `perfect` = schema valid and content 1.0.
* Main task result = mean score over the 73 cases (failed or timed-out cases count as 0 and are listed), with a 95% confidence half-width.
* Same prompt for every model; no examples from the test set in the prompt.

Cases, prompt, grader and the full 7-model run with raw responses are in the public repository linked from the write-up. Content id of the cases (sha256 of the sorted records): `a12787bb7dfa021192509942198d2b714e500cd8969215ec1e070918da6e1e29`.


In [ ]:
# Cell 1: cases (embedded, exact bytes of cases/CASES.json; no network, no model call)
import base64, gzip, hashlib, json, math
import pandas as pd
from importlib.metadata import version as _package_version
assert _package_version("kaggle-benchmarks") == "0.6.1", "Runtime must provide kaggle-benchmarks==0.6.1 before running"
import kaggle_benchmarks as kbench
assert kbench.__version__ == "0.6.1", "Loaded Kaggle Benchmarks SDK must be 0.6.1"

CASES_B64 = """..."""

_raw = gzip.decompress(base64.b64decode(CASES_B64))
assert hashlib.sha256(_raw).hexdigest() == "58603636de7b52b7f75edc1382de15c366ca249fad296a1dce351e22df207f96", "CASES.json bytes changed"
cases = pd.DataFrame(json.loads(_raw))
cases = cases[["case_id", "tier", "exceptions", "orders_json", "statement_json", "expected_json"]]
cases.index = pd.Index(cases["case_id"].tolist(), name=None)   # stable cache key per case (index = case_id)
assert cases.index.is_unique and len(cases) == 73
CASES_SHA = hashlib.sha256(cases.sort_values("case_id")[["case_id", "orders_json", "statement_json", "expected_json"]]
                           .to_json(orient="records").encode()).hexdigest()
assert CASES_SHA == "a12787bb7dfa021192509942198d2b714e500cd8969215ec1e070918da6e1e29", CASES_SHA
print("cases", len(cases), "| by tier", cases["tier"].value_counts().to_dict(), "| content sha256", CASES_SHA[:12])


In [ ]:
# Cell 2: prompt (identical to prompt/prompt.py, sha256 6ebe0f794a9601bf1ae18c531b4712eb8a8e3b4cfff385ca0c9730302d694fc3)
"""Prompt shown to every model. Same text for all models; no examples from the test set."""
import json

RULES = """You are reconciling a Brazilian merchant's Pix bank statement against its order list.

Rules:
1. Each statement line has an e2e_id. If the same e2e_id appears more than once, it is the same transfer exported twice: count it once.
2. Link a payment to an order by txid when present; otherwise by an order id written in the description. Never link by amount or payer name alone.
3. Amounts may be written as "R$ 1.234,56", "1.234,56" or "1234.56". A refund line has a negative amount and a refers_to field with the e2e_id of the payment it returns.
4. Times may be in UTC ("Z") or in -03:00. Compare instants, not clock text.
5. Status of each order, using the amount received in distinct transfers:
   - "unpaid": no linked payment.
   - "paid": exactly the order amount received, at or before expires_at.
   - "late_paid": exactly the order amount received, after expires_at.
   - "underpaid": less than the order amount received.
   - "overpaid": more than the order amount received (for example, two different transfers).
   - "refunded": the linked payment was fully returned by a refund line.
6. "payments" lists every distinct e2e_id linked to the order, including refund lines, sorted.
7. "orphans" lists distinct e2e_ids of payment lines that cannot be linked to any order, sorted.
8. Refunds: an order is refunded only when the refunds linked to it add up to exactly the total of its linked payments. A partial refund does not change the status; the refund line is still listed in "payments".
9. If more than one status could apply, use the first that applies in this order: unpaid, refunded, underpaid, overpaid, late_paid, paid. A payment exactly at expires_at is on time.
10. Every order from the order list must appear once in "orders"; do not add other ids.
11. Status uses the gross amount received in payment lines; a partial refund is not subtracted. created_at is not used by any rule.
12. When several transfers are linked to one order, compare the time of the latest one with expires_at.
13. "orphans" contains payment lines only: a refund line is never an orphan, even if its refers_to points to an orphan payment or to nothing.

Answer with exactly one JSON object and nothing else (no text before or after; one Markdown json code block around it is accepted). Exact schema, no other keys:
{"orders": {"<order_id>": {"status": "...", "payments": ["..."]}}, "orphans": ["..."]}"""


def build_prompt(orders_json: str, statement_json: str) -> str:
    orders = json.loads(orders_json)
    statement = json.loads(statement_json)
    return (
        RULES
        + "\n\nOrders:\n" + json.dumps(orders, ensure_ascii=False, indent=1)
        + "\n\nStatement:\n" + json.dumps(statement, ensure_ascii=False, indent=1)
    )


In [ ]:
# Cell 3: deterministic grader v4 (grader/grader_v4.py, sha256 7752c1e84ee3ca6b8bba34e67007dc1c63ab8b81a92b557b77f55f3fcb283ef2, minus its `from __future__` line)
"""Deterministic grading. No LLM judge.

Version 4 (2026-10-01): every order in the answer, including invented ones, is validated structurally
before the content comparison (REVISAO-K2 of the Codex, 05h40). Order ids must be strings, also when
grade() receives a dict directly.

Version 3 (2026-10-01):
- The whole answer must be one JSON object. The only wrapper allowed is a single Markdown code fence
  around the entire answer (```json ... ``` or ``` ... ```). Text before/after, a top-level array,
  two documents, duplicate keys and NaN/Infinity are rejected.
- Schema is exact: top level {"orders", "orphans"}; each order {"status", "payments"}.
- Three numbers are reported, never merged silently:
    content_score  points earned on content, in [0, 1] (as in v2);
    schema_valid   True only if the answer follows the exact schema;
    score          the value the benchmark publishes = content_score if schema_valid else 0.0.
  perfect = schema_valid and content_score == 1 and no missing or extra order.
"""

import json
import re

STATUSES = ("paid", "unpaid", "underpaid", "overpaid", "late_paid", "refunded")
TOP_KEYS = {"orders", "orphans"}
ORDER_KEYS = {"status", "payments"}

_FENCE = re.compile(r"\A```(?:json|JSON)?[ \t]*\r?\n(.*)\r?\n```\Z", re.S)


class _DupKey(Exception):
    pass


def _no_dup_pairs(pairs):
    keys = [k for k, _ in pairs]
    if len(keys) != len(set(keys)):
        raise _DupKey(sorted({k for k in keys if keys.count(k) > 1}))
    return dict(pairs)


def _no_constants(name):
    raise ValueError("non_finite_constant:" + name)


def parse_answer(text_or_obj):
    """Returns (obj, errors). obj is the parsed JSON value of the WHOLE answer, or None."""
    if isinstance(text_or_obj, dict):
        return text_or_obj, []
    if hasattr(text_or_obj, "model_dump"):
        return text_or_obj.model_dump(), []
    text = "" if text_or_obj is None else str(text_or_obj)
    text = text.strip()
    if not text:
        return None, ["empty_answer"]
    m = _FENCE.match(text)
    if m:
        text = m.group(1).strip()
        if "```" in text:
            return None, ["more_than_one_fence"]
    try:
        obj = json.loads(text, object_pairs_hook=_no_dup_pairs, parse_constant=_no_constants)
    except _DupKey as e:
        return None, ["duplicate_json_keys:" + ",".join(map(str, e.args[0]))]
    except ValueError as e:  # JSONDecodeError is a ValueError: residual text, two documents, bad syntax
        msg = str(e)
        if msg.startswith("non_finite_constant:"):
            return None, [msg]
        if "Extra data" in msg:
            return None, ["text_after_json_or_two_documents"]
        return None, ["invalid_json_or_text_around_it"]
    except RecursionError:
        return None, ["invalid_json_too_deep"]
    return obj, []


def _id_list(v, where, errors):
    """A valid id list is a list of strings without repeats. Returns sorted list or None."""
    if not isinstance(v, list):
        errors.append(f"{where}:not_a_list")
        return None
    if not all(isinstance(x, str) for x in v):
        errors.append(f"{where}:non_string_id")
        return None
    if len(v) != len(set(v)):
        errors.append(f"{where}:repeated_id")
        return None
    return sorted(v)


def _check_order(oid, g, schema_errors):
    """Structural validation of ONE order value, expected or not. Returns (status, sorted payments or None)."""
    if not isinstance(g, dict):
        schema_errors.append(f"order {oid}:not_object")
        return None, None
    if not all(isinstance(k, str) for k in g.keys()):
        schema_errors.append(f"order {oid}:non_string_key")
    ks = set(map(str, g.keys()))
    if ks - ORDER_KEYS:
        schema_errors.append(f"order {oid}:extra_keys:" + ",".join(sorted(ks - ORDER_KEYS)))
    for k in sorted(ORDER_KEYS - ks):
        schema_errors.append(f"order {oid}:{k}_missing")
    status = g.get("status")
    if "status" in g and not (isinstance(status, str) and status in STATUSES):
        schema_errors.append(f"order {oid}:invalid_status")
    pays = _id_list(g["payments"], f"order {oid}.payments", schema_errors) if "payments" in g else None
    return status, pays


def grade(answer, expected_json: str) -> dict:
    """content_score: each expected order earns 0.5 for the exact status and 0.5 for the exact list of
    linked e2e_ids; orphans earn 1 if exactly right; each extra order adds 1 to the denominator.
    v4: EVERY order present in the answer (expected or invented) is validated structurally first;
    a malformed invented order makes schema_valid False (published 0). A well-formed invented order
    only costs content points. schema_errors make schema_valid False; content_errors only cost points."""
    exp = json.loads(expected_json)
    ans, parse_errors = parse_answer(answer)
    schema_errors = list(parse_errors)
    content_errors: list[str] = []
    detail = {}
    base = {"score": 0.0, "content_score": 0.0, "schema_valid": False, "parsed": False, "perfect": False,
            "orphans_ok": False, "extra_orders": [], "missing_orders": [],
            "schema_errors": schema_errors, "content_errors": content_errors, "errors": [], "detail": detail}
    if not isinstance(ans, dict):
        if ans is not None:
            schema_errors.append("top_level_not_object")
        base["errors"] = schema_errors + content_errors
        return base
    base["parsed"] = True
    if not all(isinstance(k, str) for k in ans.keys()):
        schema_errors.append("top_level:non_string_key")
    keys = set(map(str, ans.keys()))
    if keys - TOP_KEYS:
        schema_errors.append("extra_top_level_keys:" + ",".join(sorted(keys - TOP_KEYS)))
    got_orders = ans.get("orders")
    if not isinstance(got_orders, dict):
        schema_errors.append("orders:missing_or_not_object")
        got_orders = {}
    # 1) structure of every order in the answer, independent of the gold
    checked = {}
    for oid, g in got_orders.items():
        if not isinstance(oid, str):
            schema_errors.append(f"order {oid!r}:non_string_id")
        checked[str(oid)] = _check_order(str(oid), g, schema_errors)
    # 2) content against the gold, expected orders only
    points = 0.0
    missing = []
    for oid, e in exp["orders"].items():
        if oid not in checked:
            content_errors.append(f"order {oid}:missing")
            missing.append(oid)
            status, pays = None, None
        else:
            status, pays = checked[oid]
        st_ok = isinstance(status, str) and status == e["status"]
        pay_ok = pays is not None and pays == e["payments"]
        points += 0.5 * st_ok + 0.5 * pay_ok
        detail[oid] = {"expected": e["status"], "got": status if isinstance(status, (str, type(None))) else repr(status),
                       "status_ok": st_ok, "payments_ok": pay_ok}
    extra = sorted(k for k in checked if k not in exp["orders"])
    if extra:
        content_errors.append("extra_orders:" + ",".join(extra))
    if "orphans" not in ans:
        schema_errors.append("orphans:missing")
        orph = None
    else:
        orph = _id_list(ans["orphans"], "orphans", schema_errors)
    orph_ok = orph is not None and orph == exp["orphans"]
    points += 1.0 * orph_ok
    n = len(exp["orders"]) + 1 + len(extra)
    content = points / n
    valid = not schema_errors
    base.update(content_score=content, schema_valid=valid, score=content if valid else 0.0,
                orphans_ok=orph_ok, extra_orders=extra, missing_orders=missing,
                perfect=(valid and content == 1.0 and not content_errors),
                errors=schema_errors + content_errors)
    return base


In [ ]:
# Cell 4: tasks. One sub-task per case (not stored) and one main task (the benchmark page).
import time
RUN_TIMEOUT_S = 600      # per case; reasoning models take 40 to 300 s on hard cases
N_JOBS = 4               # parallel cases
RETRY_DELAY_S = 20       # before the second (and last) attempt of the cases that failed on transport

@kbench.task(name="pixrecon_case", store_task=False)
def pixrecon_case(llm, case_id: str, tier: str, exceptions: str, orders_json: str,
                  statement_json: str, expected_json: str) -> dict:
    """One case: prompt the model once, grade the answer deterministically."""
    response = llm.prompt(build_prompt(orders_json, statement_json))
    g = grade(response, expected_json)   # never raises on malformed answers
    text = response if isinstance(response, str) else repr(response)
    return {
        "case_id": case_id, "tier": tier, "exceptions": exceptions,
        "score": float(g["score"]), "content_score": float(g["content_score"]),
        "schema_valid": bool(g["schema_valid"]), "perfect": bool(g["perfect"]),
        "schema_errors": list(g.get("schema_errors", [])), "content_errors": list(g.get("content_errors", [])),
        "response_chars": len(text), "response_sha256": hashlib.sha256(text.encode()).hexdigest(),
    }


# SDK 0.6.1 Run.error_message stores traceback.format_exc().
# Only a known final transport exception can receive a second attempt.
_TRANSPORT_ERRORS = {
    "httpx.ConnectError", "httpx.ReadError", "httpx.WriteError",
    "httpx.ConnectTimeout", "httpx.ReadTimeout", "httpx.WriteTimeout",
    "httpx.RemoteProtocolError", "openai.APIConnectionError", "openai.APITimeoutError",
    "requests.exceptions.ConnectionError", "requests.exceptions.ConnectTimeout",
    "requests.exceptions.ReadTimeout", "ConnectionResetError", "BrokenPipeError",
}


def _retry_reason(error_message):
    text = str(error_message or "")
    if not text or re.search(
        r"\b(?:429|401|403|unknown|quota|resource_exhausted|insufficient_quota|"
        r"ratelimiterror|authenticationerror|permissiondeniederror|unauthorized|forbidden)\b",
        text, re.IGNORECASE,
    ):
        return "NO_RETRY:QUOTA_AUTH_RATE_LIMIT_OR_UNKNOWN"
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    exception_type = lines[-1].partition(":")[0] if lines else ""
    if exception_type in _TRANSPORT_ERRORS:
        return "TRANSPORT:" + exception_type
    return "NO_RETRY:UNCLASSIFIED"


def _disable_client_retries(llm):
    # ModelProxy defaults to this adapter in SDK 0.6.1. Do not guess other APIs.
    from kaggle_benchmarks.actors.llms import OpenAI as _ProxyOpenAI
    if not isinstance(llm, _ProxyOpenAI):
        raise RuntimeError("Unverified proxy adapter: stop before model calls")
    llm.client = llm.client.with_options(max_retries=0)
    if llm.client.max_retries != 0:
        raise RuntimeError("Proxy client retries must be disabled before model calls")


def _evaluate_once(llm, rows):
    # nested evaluations always run with max_attempts=1 (SDK rule); the retry is done explicitly below
    runs = pixrecon_case.evaluate(llm=[llm], evaluation_data=rows, n_jobs=N_JOBS, timeout=RUN_TIMEOUT_S,
                                  max_attempts=1, on_failure="continue", remove_run_files=True)
    done = {r.params["case_id"]: r.result for r in runs.completed_runs if isinstance(r.result, dict)}
    errors = {r.params.get("case_id"): _retry_reason(r.error_message) for r in runs.errored_runs}
    return done, errors


@kbench.task(
    name="PixRecon Pix Reconciliation",
    description="Reconcile a messy Brazilian Pix bank statement against an order list (73 cases: duplicates, refunds, "
                "late, partial, over and orphan payments, UTC vs -03:00). Strict JSON answer, deterministic grader.",
)
def pixrecon(llm, cases: pd.DataFrame) -> tuple[float, float]:
    _disable_client_retries(llm)
    with kbench.client.enable_cache():
        done, errors = _evaluate_once(llm, cases)
        retry_ids = [cid for cid in cases["case_id"]
                     if cid not in done and errors.get(cid, "").startswith("TRANSPORT:")]
        if retry_ids:   # second and last attempt: identified transport errors only
            print(f"second and last attempt for {len(retry_ids)} case(s): {retry_ids}")
            time.sleep(RETRY_DELAY_S)
            done2, errors2 = _evaluate_once(llm, cases.loc[retry_ids])
            done.update(done2)
            errors.update(errors2)
    rows = []
    for cid, tier in zip(cases["case_id"], cases["tier"]):
        d = done.get(cid)
        if d is not None:
            rows.append({"case_id": cid, "tier": tier, "score": d["score"], "schema_valid": d["schema_valid"],
                         "perfect": d["perfect"], "status": "scored"})
        else:
            rows.append({"case_id": cid, "tier": tier, "score": 0.0, "schema_valid": False,
                         "perfect": False, "status": "failed_or_missing"})
    g = pd.DataFrame(rows)
    n = len(g)
    mean = float(g["score"].mean())
    sd = float(g["score"].std(ddof=1)) if n > 1 else 0.0
    ci95 = float(1.96 * sd / math.sqrt(n)) if n > 1 else 0.0
    print(f"PixRecon | cases {n} | scored {int((g.status == 'scored').sum())} | failed {int((g.status != 'scored').sum())}")
    print(f"mean score {mean:.4f} (95% CI +/- {ci95:.4f}) | schema valid {int(g.schema_valid.sum())}/{n} | perfect {int(g.perfect.sum())}/{n}")
    print(g.groupby("tier")["score"].agg(["count", "mean"]).round(4))
    failed = g.loc[g.status != "scored", "case_id"].tolist()
    if failed:
        print("counted as 0 (non-retryable or failed after second attempt):", failed)
        for cid in failed:
            print("  ", cid, "|", errors.get(cid, "no error message"))
    return mean, ci95


In [ ]:
# Cell 5: run the benchmark on the model under test (kbench.llm = the notebook's default model)
run = pixrecon.run(kbench.llm, cases)
run


In [ ]:
%choose pixrecon
